# ⛰️ Day 25 · Is this a map? Chimgan in 3D

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2025%20Is%20this%20a%20map/tutorial/day25_chimgan_in_3d.ipynb)

A map is usually flat, seen from straight above. But what if we cut a piece out of the Earth, like a slice of cake, and look at it from the side? Is that still a map?

In this notebook you will:
1. load the **heights** of Chimgan and the Charvak reservoir from satellite elevation data,
2. draw them as a normal flat map,
3. turn them into a **3D surface** that you can rotate with your mouse,
4. learn why mountains in 3D maps are usually **stretched** up,
5. cut out a **cake slice** of the mountains,
6. lay a **satellite picture** on top,
7. save your 3D map as a web page you can share,
8. draw the same mountains as **lines only**, and ask: is this still a map?

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

We use the same free tools as on Day 13 and Day 23: they search the **Microsoft Planetary Computer**, a big free library of satellite data, and load only the piece we need. No account is needed.

The heights come from the **Copernicus DEM**: a satellite measured the height of every place on Earth, every 30 m. (DEM means *digital elevation model*: a picture where every pixel is a height instead of a colour.) We load it with **100 m** pixels, so the 3D map is quick to draw.

`BOX` is the box around Chimgan and Charvak: west, south, east and north, in degrees.

In [ ]:
!pip -q install pystac-client planetary-computer odc-stac

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, LinearSegmentedColormap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pystac_client, planetary_computer, odc.stac

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

BOX = [69.92, 41.43, 70.22, 41.70]      # west, south, east, north: Chimgan and Charvak

def load_heights(box, resolution=100):
    lng = (box[0] + box[2]) / 2
    utm = f"EPSG:{32600 + int((lng + 180) // 6) + 1}"      # the UTM zone, in metres
    items = catalog.search(collections=["cop-dem-glo-30"], bbox=box).item_collection()
    return odc.stac.load(items, bands=["data"], bbox=box, crs=utm, resolution=resolution).isel(time=0)["data"]

dem = load_heights(BOX)
heights = dem.values.astype(float)                       # metres above sea level
x_km = (dem.x.values - dem.x.values.min()) / 1000        # km from the west side
y_km = (dem.y.values - dem.y.values.min()) / 1000        # km from the south side

# colours from low to high: green valleys, brown slopes, white peaks
LAND = [[0, "#5f7f4f"], [0.25, "#a3a86b"], [0.5, "#c9a46f"], [0.75, "#8d6b4f"], [0.9, "#d9d4cc"], [1, "#ffffff"]]
BACK = "#f4efe6"

print(heights.shape[1], "×", heights.shape[0], "pixels,", round(x_km.max()), "×", round(y_km.max()), "km")
print("From", round(heights.min()), "to", round(heights.max()), "m. Ready! ✅")

## Step 1 · A flat map first

This is the map you know: seen from straight above, colours for heights, and shadows as if the sun shone from the north-west (**hillshade**). The shadows already make our eyes see mountains.

In [ ]:
land_cmap = LinearSegmentedColormap.from_list("land", [(p, c) for p, c in LAND])
shaded = LightSource(azdeg=315, altdeg=45).shade(heights, cmap=land_cmap, vert_exag=2,
                                                 blend_mode="soft", dx=100, dy=100)

fig, ax = plt.subplots(figsize=(9, 10))
ax.imshow(shaded, extent=[0, x_km.max(), 0, y_km.max()])
ax.set_xlabel("km"); ax.set_ylabel("km")
plt.show()

Big Chimgan is the white massif in the south. The flat green floor in the north is **Charvak**: the elevation data sees only the top of the water.

## Step 2 · Your first 3D map

**plotly** draws a `Surface`: for every pixel, it puts a point at its height and joins the points into a skin.

We give it the heights **in km**, like the sides, so that nothing is stretched. `aspectmode="data"` means "keep the real proportions".

**Drag** the map with your mouse to rotate it, **scroll** to zoom.

In [ ]:
fig = go.Figure(go.Surface(x=x_km, y=y_km, z=heights / 1000, colorscale=LAND, showscale=False))
fig.update_layout(scene=dict(aspectmode="data"), height=600, margin=dict(l=0, r=0, t=0, b=0))
fig.show()

These are real proportions: Chimgan rises about 2.4 km above Charvak, and our box is 25 km wide. The mountains look smaller than they feel when you climb them!

## Step 3 · Stretch it up

Almost every 3D map you've seen is **stretched up**: the heights are multiplied by 2, 3 or more. This is called **vertical exaggeration**. It shows the shape of the land better, but it isn't the real Earth.

Step 3 draws the same mountains side by side: real (×1) and stretched three times (×3).

In [ ]:
def scene(camera_distance=2.1):
    hide = dict(visible=False)
    return dict(aspectmode="data", xaxis=hide, yaxis=hide, zaxis=hide,
                camera=dict(eye=dict(x=-0.55 * camera_distance, y=-0.8 * camera_distance, z=0.45 * camera_distance)))

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=["×1, real", "×3, stretched"])
for col, stretch in [(1, 1), (2, 3)]:
    fig.add_trace(go.Surface(x=x_km, y=y_km, z=heights / 1000 * stretch,
                             colorscale=LAND, showscale=False), row=1, col=col)
fig.update_layout(scene=scene(3.8), scene2=scene(3.8), height=500, margin=dict(l=0, r=0, t=30, b=0))
fig.show()

In the rest of the notebook we use `STRETCH = 2`: enough to see the valleys, not so much that the mountains look like needles. Try other numbers!

## Step 4 · Cut a cake slice

A 3D surface floating in the air looks strange. Let's give it **walls**, like a slice of cake, or the block diagrams in geography textbooks.

Each wall is one more `Surface`: along one edge of the box, from a flat **bottom** up to the heights of that edge.

In [ ]:
STRETCH = 2
WALL = [[0, "#6e4a2e"], [1, "#c99a68"]]          # dark brown at the bottom, light at the top

def add_walls(fig, x, y, z, bottom):
    edges = [(x, np.full_like(x, y[0]), z[0]),        # the 4 edges of the box
             (x, np.full_like(x, y[-1]), z[-1]),
             (np.full_like(y, x[0]), y, z[:, 0]),
             (np.full_like(y, x[-1]), y, z[:, -1])]
    for ex, ey, top in edges:
        wz = np.vstack([np.full_like(top, bottom), top])
        fig.add_trace(go.Surface(x=np.vstack([ex, ex]), y=np.vstack([ey, ey]), z=wz,
                                 surfacecolor=wz, colorscale=WALL, showscale=False))

def make_cake(dem, stretch=STRETCH):
    x = (dem.x.values - dem.x.values.min()) / 1000
    y = (dem.y.values - dem.y.values.min()) / 1000
    z = dem.values.astype(float) / 1000 * stretch
    fig = go.Figure(go.Surface(x=x, y=y, z=z, colorscale=LAND, showscale=False,
                               lighting=dict(ambient=0.5, diffuse=0.8, roughness=0.9, specular=0.1)))
    add_walls(fig, x, y, z, bottom=z.min() - 0.8)
    fig.update_layout(scene=scene(), paper_bgcolor=BACK, height=650, margin=dict(l=0, r=0, t=0, b=0))
    return fig

make_cake(dem).show()

## Step 5 · Lay a satellite picture on top

Colours for heights are nice, but what does Chimgan really look like? Let's take a **Sentinel-2** picture (like on Day 23) and lay it on the 3D surface.

We use **30 August 2025**, a clear day from Day 23. `like=dem` loads the picture on exactly the same 100 m pixels as the heights, so every height gets its own colour. `resampling="average"` makes each 100 m pixel the average of the 10 m pixels inside it.

In [ ]:
items = catalog.search(collections=["sentinel-2-l2a"], bbox=BOX, datetime="2025-08-30").item_collection()
photo = odc.stac.load(items, bands=["B04", "B03", "B02"], like=dem, groupby="solar_day",
                       resampling="average").isel(time=0)

rgb = np.dstack([photo[b].values for b in ["B04", "B03", "B02"]])
rgb = ((rgb - 1000) / 10000 / 0.28).clip(0, 1) ** 0.9            # numbers → reflectance → brightness
colours = [f"rgb({r:.0f},{g:.0f},{b:.0f})" for r, g, b in (rgb * 255).reshape(-1, 3)]

plt.figure(figsize=(6, 7)); plt.imshow(rgb); plt.axis("off"); plt.show()

A `Surface` can only colour by one number, so for a picture we use a `Mesh3d`: a skin made of **triangles**, where every corner can have its own colour. Every square of 4 pixels becomes 2 triangles.

In [ ]:
z = heights / 1000 * STRETCH
rows, cols = z.shape
corner = np.arange(rows * cols).reshape(rows, cols)          # a number for every pixel
a, b = corner[:-1, :-1].ravel(), corner[:-1, 1:].ravel()     # top-left, top-right
c, d = corner[1:, :-1].ravel(), corner[1:, 1:].ravel()       # bottom-left, bottom-right
xx, yy = np.meshgrid(x_km, y_km)

fig = go.Figure(go.Mesh3d(x=xx.ravel(), y=yy.ravel(), z=z.ravel(),
                          i=np.r_[a, b], j=np.r_[b, d], k=np.r_[c, c],   # 2 triangles per square
                          vertexcolor=colours, lighting=dict(ambient=0.75, diffuse=0.45, specular=0, roughness=1)))
add_walls(fig, x_km, y_km, z, bottom=z.min() - 0.8)
fig.update_layout(scene=scene(), paper_bgcolor=BACK, height=650, margin=dict(l=0, r=0, t=0, b=0))
fig.show()

Now the reservoir is blue, and you can see the villages along its shore and the roads up the valleys. Rotate it to look at Charvak from Chimgan!

## Step 6 · Save your 3D map

`write_html` saves the 3D map as a **web page**. Anyone can open it in a browser and rotate it, with no Python. Send it to a friend!

In [ ]:
fig.write_html("chimgan_3d.html", include_plotlyjs="cdn")

try:
    from google.colab import files
    files.download("chimgan_3d.html")
except ImportError:
    print("Saved chimgan_3d.html")

## Step 7 · Is this a map?

Now the same mountains with almost nothing: one **line** for every row of pixels, from north to south, each line lifted by its heights. Each line hides the lines behind it, so the lines in front look closer.

There's no colour, no satellite, no labels. Is it still a map? It has a real place, real heights and a real north. You decide!

In [ ]:
LINES = 70
step = len(heights) // LINES
gap = 150                                  # metres between two lines
stretch = 1.5

fig, ax = plt.subplots(figsize=(9, 11), facecolor="#111111")
ax.set_facecolor("#111111")
for n, row in enumerate(range(0, len(heights), step)):      # row 0 is the north, at the back
    line = (heights[row] - heights.min()) * stretch - n * gap
    ax.fill_between(x_km, line, line.min() - 5000, color="#111111", zorder=n)
    ax.plot(x_km, line, color="#f2efe8", linewidth=1.1, zorder=n + 0.5)
ax.set_axis_off()
plt.savefig("chimgan_lines.png", dpi=200, facecolor="#111111", bbox_inches="tight")
plt.show()

try:
    from google.colab import files
    files.download("chimgan_lines.png")
except ImportError:
    print("Saved chimgan_lines.png")

## Step 8 · Your own mountain

Choose another place from the list, or add your own box: draw it on [bboxfinder.com](http://bboxfinder.com) and copy the four numbers (west, south, east, north). Then run the cell.

In [ ]:
PLACES = {
    "Chimgan":   [69.92, 41.43, 70.22, 41.70],
    "Zaamin":    [68.25, 39.45, 68.65, 39.75],     # the Turkestan range
    "Baysun":    [66.85, 38.10, 67.25, 38.40],     # the Baysuntau
    "Nurata":    [65.90, 40.45, 66.40, 40.75],     # the Nurata mountains
    "Chatkal":   [70.20, 41.65, 70.60, 41.95],
}
place = "Zaamin"

my_dem = load_heights(PLACES[place])
print(place, "from", round(float(my_dem.min())), "to", round(float(my_dem.max())), "m")
make_cake(my_dem).show()

## 🎉 Done!

You turned satellite heights into a 3D map, stretched it, cut a cake slice, laid a satellite picture on top, and drew the same mountains with only lines.

Share your map with **#30DayMapChallenge** and **#30DayMapChallengeUzbekistan**, and please credit:
> Heights: Copernicus DEM GLO-30. Picture: contains modified Copernicus Sentinel-2 data (2025). Both accessed through the Microsoft Planetary Computer.